Day 4: Frontier LLM — Vietnamese Price Prediction (Zero-shot)
#
Test 3 OpenAI models (200 items each):
- gpt-4o-mini
- gpt-5-nano
- gpt-5-mini
#
**Dataset:** SeanSunny/items_tv_v6 filtered <= 1M VND (test set only)


In [1]:
import sys
import json
from pathlib import Path

from dotenv import load_dotenv
from litellm import completion

DAY4 = Path.cwd()
sys.path.insert(0, str(DAY4.parent))
from pricer_vi.items import Item
from pricer_vi.evaluator import evaluate


In [2]:
load_dotenv(override=True)
DATASET = "SeanSunny/items_tv_v6"
MAX_PRICE = 1_000_000
SIZE = 200
WORKERS = 3


In [3]:
print("Loading data...")
_, _, test_items = Item.from_hub(DATASET)
test_items = [it for it in test_items if 0 < it.price <= MAX_PRICE]
print(f"Test items: {len(test_items):,} (evaluate {SIZE})")


Loading data...
Test items: 3,872 (evaluate 200)


In [19]:
SYSTEM_PROMPT1 = (
    "You are a price estimation expert for Vietnamese e-commerce products. "
    "Estimate the price in VND based on the product description. "
    "All products are priced under 1,000,000 VND. "
    "Respond with ONLY the number (integer), no currency symbol, no explanation."
)

SYSTEM_PROMPT = (
    "Bạn là một chuyên gia định giá các sản phẩm thương mại điện tử tại Việt Nam. "
    "Hãy ước tính giá trị sản phẩm bằng đơn vị VNĐ dựa trên mô tả sản phẩm. "
    "Các sản phẩm luôn có mức giá dưới 1.000.000 (1 triệu VNĐ). "
    "Chỉ phản hồi DUY NHẤT một con số (số nguyên), không kèm ký hiệu tiền tệ, không giải thích gì thêm."
)

def messages_for(item):
    return [
        {"role": "system", "content": SYSTEM_PROMPT1},
        {"role": "user", "content": item.summary},
    ]


In [20]:
print("\n--- Testing prompt ---")
sample = test_items[0]
print(f"Product: {sample.title}")
print(f"Actual price: {sample.price:,} VND")
response = completion(model="openai/gpt-4o-mini", messages=messages_for(sample))
print(f"LLM response: {response.choices[0].message.content}")



--- Testing prompt ---
Product: Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L HULKER NEW màu Xanh quân đội| Index Living Mall
Actual price: 479,400 VND
LLM response: 450000


In [17]:
messages_for(sample)

[{'role': 'system',
  'content': 'You are a price estimation expert for Vietnamese e-commerce products. Estimate the price in VND based on the product description. All products are priced under 1,000,000 VND. Respond with ONLY the number (integer), no currency symbol, no explanation.'},
 {'role': 'user',
  'content': 'Tiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  \nDanh mục: Hộp đựng, Thùng lưu trữ  \nThương hiệu: Index Living Mall  \nMô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  \nThông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống thấm nước.'}]

In [16]:
print(sample)

title='Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L HULKER NEW màu Xanh quân đội| Index Living Mall' category='Nhà Cửa - Đời Sống' price=479400 full=None brand=None summary='Tiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  \nDanh mục: Hộp đựng, Thùng lưu trữ  \nThương hiệu: Index Living Mall  \nMô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  \nThông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống thấm nước.' prompt='Sản phẩm này giá bao nhiêu?\n\nTiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  \nDanh mục: Hộp đựng, Thùng lưu trữ  \nThương hiệu: Index Living Mall  \nMô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  \nThông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống thấm nước.\n\nGiá: 479400' id=None


In [21]:
def gpt_4o_mini(item):
    response = completion(model="openai/gpt-4o-mini", messages=messages_for(item))
    return response.choices[0].message.content

def gpt_5_nano(item):
    response = completion(model="openai/gpt-5-nano", messages=messages_for(item))
    return response.choices[0].message.content

def gpt_5_mini(item):
    response = completion(model="openai/gpt-5-mini", messages=messages_for(item))
    return response.choices[0].message.content


In [22]:
print("\n\n" + "#" * 60)
print("# GPT-4o-mini (200 items)")
print("#" * 60)
results_4o_mini = evaluate(gpt_4o_mini, test_items, size=SIZE, workers=WORKERS)




############################################################
# GPT-4o-mini (200 items)
############################################################


  0%|          | 0/200 [00:00<?, ?it/s]

370,600 251,000 163,000 390,000 6,000 115,000 66,000 109,000 100,000 102,000 181,000 205,000 223,000 98,000 26,000 61,000 120,000 301,000 240,000 627,000 316,500 200,000 527,000 15,000 550,000 451,000 46,000 235,000 365,000 101,000 279,000 171,000 282,000 461,000 674,160 61,000 310,000 81,000 94,000 15,000 455,000 226,500 99,999 312,000 290,001 30,000 5,000 85,000 334,000 301,000 86,750 385,000 762,000 129,000 675,000 531,000 203,442 545,750 425,000 300,000 25,000 70,000 87,000 365,000 400,000 25,000 391,000 33,535 270,000 275,000 215,000 331,000 161,000 70,000 661,000 231,000 100,000 251,000 101,000 549,000 5,000 160,000 540,000 100,000 701,000 207,700 301,000 351,000 241,000 251,000 301,000 651,000 697,000 240,000 351,000 261,000 75,000 350,000 371,000 510,000 240,000 181,000 122,500 451,000 113,000 528,000 500,000 311,500 160,000 200,000 107,300 139,000 199,000 480,000 295,000 25,000 471,000 69,000 146,000 35,000 9,000 408,000 221,000 84,000 74,000 378,830 361,000 51,000 104,000 280

In [23]:
print("\n\n" + "#" * 60)
print("# GPT-5-nano (200 items)")
print("#" * 60)
results_5_nano = evaluate(gpt_5_nano, test_items, size=SIZE, workers=WORKERS)




############################################################
# GPT-5-nano (200 items)
############################################################


  0%|          | 0/200 [00:00<?, ?it/s]

159,400 100,000 73,000 139,000 73,000 65,000 36,000 79,000 10,000 51,000 211,000 54,000 228,000 48,000 195,000 11,000 19,000 201,000 39,000 367,000 83,500 200,000 76,000 105,000 250,000 151,000 76,000 84,000 316,000 9,000 21,000 71,000 81,000 361,000 223,160 91,000 210,000 80,000 204,000 36,000 205,000 75,500 129,999 61,000 90,001 260,000 20,000 15,000 11,000 350,000 85,750 166,000 311,000 109,000 105,000 80,000 3,442 245,750 574,000 390,000 75,000 300,000 23,000 365,000 90,000 26,000 161,000 196,465 70,000 54,000 85,000 111,000 171,000 70,000 161,000 180,000 30,000 251,000 29,000 549,000 46,000 40,000 51,000 40,000 450,000 58,700 201,000 700,000 90,000 100,000 101,000 451,000 297,000 570,000 151,000 231,000 45,000 380,000 171,000 259,000 140,000 20,000 71,500 351,000 127,000 228,000 200,000 160,500 60,000 130,000 213,700 12,000 300,000 350,000 105,000 20,000 171,000 50,000 76,000 35,000 91,000 157,000 30,000 21,000 277,000 378,830 301,000 81,000 304,000 150,000 170,000 3,000 200,000 4

In [24]:
print("\n\n" + "#" * 60)
print("# GPT-5-mini (200 items)")
print("#" * 60)
results_5_mini = evaluate(gpt_5_mini, test_items, size=SIZE, workers=WORKERS)




############################################################
# GPT-5-mini (200 items)
############################################################


  0%|          | 0/200 [00:00<?, ?it/s]

259,400 121,000 59,000 39,000 6,000 186,000 64,000 42,000 70,000 32,000 51,000 125,000 228,000 17,000 104,000 60,000 120,000 50,000 65,000 427,000 134,500 260,000 427,000 120,000 250,000 100,000 37,000 35,000 565,000 50,000 9,000 29,000 52,000 210,000 173,160 61,000 110,000 49,000 395,000 136,000 155,000 73,500 50,999 112,000 30,001 190,000 10,000 45,000 34,000 101,000 56,750 116,000 211,000 199,000 124,000 20,000 226,558 124,750 225,000 115,000 375,000 70,000 7,000 265,000 20,000 75,000 40,000 166,465 130,000 124,000 85,000 31,000 31,000 170,000 110,000 31,000 51,000 100,000 99,000 149,000 46,000 340,000 70,000 140,000 350,000 58,700 201,000 151,000 61,000 100,000 50,000 200,000 97,000 591,000 151,000 31,000 146,000 450,000 141,000 110,000 110,000 59,000 28,500 151,000 164,000 228,000 300,000 110,500 11,000 79,000 252,700 12,000 249,000 129,000 56,000 22,000 29,000 199,000 25,000 5,000 161,000 57,000 20,000 61,000 326,000 148,830 161,000 21,000 434,000 129,000 251,000 12,000 300,000 6

In [25]:
print("\n\n" + "=" * 60)
print("FRONTIER LLM RESULTS (200 items each)")
print("=" * 60)

all_results = {
    "gpt-4o-mini": results_4o_mini,
    "gpt-5-nano": results_5_nano,
    "gpt-5-mini": results_5_mini,
}

for name, r in all_results.items():
    print(f"\n{name}:")
    print(f"  RMSLE: {r['rmsle']:.4f} | MAE: {r['mae']:,.0f} VND | MAPE: {r['mape']:.1f}% | R2: {r['r2']:.1f}%")

# Save results
results_path = DAY4 / "day4_frontier_llm_results.json"
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {results_path}")




FRONTIER LLM RESULTS (200 items each)

gpt-4o-mini:
  RMSLE: 0.9894 | MAE: 248,392 VND | MAPE: 167.6% | R2: -61.8%

gpt-5-nano:
  RMSLE: 0.7025 | MAE: 157,939 VND | MAPE: 87.8% | R2: 21.6%

gpt-5-mini:
  RMSLE: 0.6261 | MAE: 131,132 VND | MAPE: 64.4% | R2: 43.8%

Saved: /home/hieu0606sunny/price2026wsl/tech2ai/scraping_data_tv/Data_processing_for_Vietnamese_data/day4/day4_frontier_llm_results.json
